# Chronos-2: univariate forecasting

## (1) Introduction

[Chronos-2](https://huggingface.co/amazon/chronos-2) is Amazon's 120M-parameter, encoder-only time-series foundation model. It supports zero-shot univariate forecasting and future-known covariates, and returns probabilistic quantile forecasts.

Run the sections in order. Section (5) evaluates the configured model in the configured windows. Section (6) reuses the final configuration for historical backtests or future production forecasts.

References: [official Chronos-2 notebook](https://colab.research.google.com/drive/1IkYv2EwlToKe2uZGWlhouNp1Kh3caNwP) and [model card](https://huggingface.co/amazon/chronos-2).

## (2) General configuration

**Required input schema:** the input file or sheet must contain columns named exactly `item_id`, `timestamp`, and `target`. Extra numeric columns may be present and are used from Google Sheets only when `USE_GSHEET_SHEET_COVARIATES = True`.

The CSV metadata-row count and the three required input-column names are fixed notebook constants, not user settings. Keep the configuration blocks below in this order.

In [ ]:
# INPUT CONFIGURATION
DATA_SOURCE = "gsheet"                                       # "csv" | "gsheet"
CSV_PATH = "data/Data - Consumo_BD.csv"
FILE_ID = "1hakzOf9fUA8dA9TLxeph4wrd3EpWUWS0jI8QbzfGBsg"     # Required for Google Sheets
SHEET_NAME = "Consumo_BD"                              # Sheet containing item_id, timestamp, target, and optional covariates
TARGET_NAME = "consumo_es_real_des"
TRANSFORMATION = "log"                                       # "log" | "log1p" | "level"

# NATURAL/BUSINESS DATE CONFIGURATION
DATA_CALENDAR = "natural"                     # "natural" (7 days) | "business" (Monday-Friday)
MISSING_METHOD = "previous_week_mean"         # "previous_week_mean" | "linear"

# COVARIATE CONFIGURATION
USE_GSHEET_SHEET_COVARIATES = False          # True adds numeric covariates from the target sheet
ADD_HOLIDAY_COVARIATES = True                # Adds holiday, pre/post-holiday, and bridge-day dummies
HOLIDAY_COUNTRY = "Spain"                    # e.g. "Spain", "United States", "France", "ES", "US"

# Set each basic covariate to True only when it should be passed to Chronos-2.
BASIC_COVARIATES_TO_INCLUDE = {
    "day_of_week": True,
    "month_sin": True,
    "month_cos": True,
    "weekend": True,
    "day_of_month": True,
    "first_natural_day_of_month": True,
    "first_business_day_of_month": True,
    "last_natural_day_of_month": True,
    "last_business_day_of_month": True,
}

# MODEL CONFIGURATION
MODEL_NAME = "amazon/chronos-2"
CHRONOS2_MAX_CONTEXT = 8192
CHRONOS2_BATCH_SIZE = 32
QUANTILE_LEVELS = [0.1, 0.5, 0.9]


## (3) Packages and generic functions

Run these cells once. This section only installs packages and defines reusable functions. Certificate verification remains enabled. By default, `truststore` lets Hugging Face use the Windows/system certificate store. If the corporate CA is not installed there, set `HUGGINGFACE_CA_BUNDLE` in the installation cell below, restart the kernel, and rerun this section.


### (3.a) Package installation

In [ ]:
# Installation, imports, and secure certificate setup
import os

# Optional corporate root/intermediate CA bundle in PEM format.
HUGGINGFACE_CA_BUNDLE = None                 # e.g. r"C:\certificates\company-ca.pem"
import re
import subprocess
import sys
from pathlib import Path

try:
    import torch
    from chronos import Chronos2Pipeline
    import truststore
    import workalendar
    from statsmodels.tsa.seasonal import seasonal_decompose
except ImportError:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "chronos-forecasting>=2.0", "pandas[pyarrow]", "truststore",
        "workalendar", "statsmodels", "openpyxl",
    ])
    import torch
    from chronos import Chronos2Pipeline
    import truststore
    import workalendar
    from statsmodels.tsa.seasonal import seasonal_decompose

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display


def configure_huggingface_certificates():
    if HUGGINGFACE_CA_BUNDLE:
        bundle = Path(HUGGINGFACE_CA_BUNDLE)
        if not bundle.is_file():
            raise FileNotFoundError(f"HUGGINGFACE_CA_BUNDLE does not exist: {bundle}")
        os.environ["SSL_CERT_FILE"] = str(bundle.resolve())
        print("Using the configured corporate CA bundle.")
    else:
        truststore.inject_into_ssl()
        print("Using the operating-system certificate store via truststore.")


def reset_huggingface_session():
    try:
        from huggingface_hub.utils import close_session
        close_session()
    except Exception:
        pass


configure_huggingface_certificates()
CHRONOS2_DEVICE_MAP = "cuda" if torch.cuda.is_available() else "cpu"
print("Chronos-2 ready. Device:", CHRONOS2_DEVICE_MAP)


### (3.b) Generic function definitions


In [ ]:
# Fixed input constants and generic data functions
CSV_SKIPROWS = 2
INPUT_DATE_COLUMN = "timestamp"
INPUT_TARGET_COLUMN = "target"
ITEM_ID_COLUMN = "item_id"
DATE_COLUMN, TARGET_COLUMN, PROCESSED_TARGET_COLUMN = "timestamp", "target", "target_proc"
NATURAL_CUTOFF_DAYS = [7, 14, 21, 28]
BUSINESS_CUTOFF_DAYS = [5, 10, 15, 20]


def coerce_numeric(values):
    values = pd.Series(values)
    if pd.api.types.is_numeric_dtype(values):
        return pd.to_numeric(values, errors="coerce")
    values = values.astype(str).str.strip().str.replace("\u00a0", "", regex=False).str.replace(" ", "", regex=False)
    has_comma = values.str.contains(",", regex=False, na=False)
    values.loc[has_comma] = (values.loc[has_comma].str.replace(".", "", regex=False)
                             .str.replace(",", ".", regex=False))
    return pd.to_numeric(values, errors="coerce")


def parse_dates(values):
    values = pd.Series(values).astype(str).str.strip()
    european = pd.to_datetime(values, format="%d/%m/%Y", errors="coerce")
    return european.fillna(pd.to_datetime(values, errors="coerce"))


def transform_target(values):
    values = pd.to_numeric(pd.Series(values), errors="coerce").to_numpy(float)
    if TRANSFORMATION == "log":
        if np.any(values[np.isfinite(values)] <= 0):
            raise ValueError("The log transformation requires strictly positive target values.")
        return np.log(values)
    if TRANSFORMATION == "log1p":
        return np.log1p(values)
    if TRANSFORMATION == "level":
        return values
    raise ValueError("TRANSFORMATION must be 'log', 'log1p', or 'level'.")


def inverse_transform(values):
    values = np.asarray(values, dtype=float)
    if TRANSFORMATION == "log":
        return np.exp(values)
    if TRANSFORMATION == "log1p":
        return np.expm1(values)
    return values


def safe_token(value):
    return re.sub(r"[^0-9A-Za-z_]+", "_", str(value).lower()).strip("_") or "event"


def detect_header(values):
    required = {ITEM_ID_COLUMN, INPUT_DATE_COLUMN, INPUT_TARGET_COLUMN}
    for position, row in enumerate(values):
        if required.issubset({str(value).strip() for value in row}):
            return position
    raise ValueError(f"The sheet must contain columns named exactly {sorted(required)}.")


def resolve_csv_path():
    candidates = [Path(CSV_PATH), Path("modelos/chronos-2") / CSV_PATH]
    for candidate in candidates:
        if candidate.is_file():
            return candidate
    raise FileNotFoundError(f"CSV_PATH was not found. Checked: {[str(path) for path in candidates]}")


def load_raw_input():
    if DATA_SOURCE == "csv":
        return pd.read_csv(resolve_csv_path(), skiprows=CSV_SKIPROWS, dtype=str)
    if DATA_SOURCE == "gsheet":
        if not FILE_ID or not SHEET_NAME:
            raise ValueError("Set FILE_ID and SHEET_NAME for Google Sheets.")
        from google.colab import auth
        from google.auth import default
        import gspread
        auth.authenticate_user()
        credentials, _ = default()
        values = gspread.authorize(credentials).open_by_key(FILE_ID).worksheet(SHEET_NAME).get_all_values()
        header_position = detect_header(values)
        return pd.DataFrame(values[header_position + 1:], columns=[str(x).strip() for x in values[header_position]])
    raise ValueError("DATA_SOURCE must be 'csv' or 'gsheet'.")


def impute_missing_history(history):
    history = history.copy().sort_values(DATE_COLUMN)
    values = history.set_index(DATE_COLUMN)[TARGET_COLUMN].astype(float)
    period = 7 if DATA_CALENDAR == "natural" else 5
    if MISSING_METHOD == "previous_week_mean":
        for position in range(len(values)):
            if pd.isna(values.iloc[position]):
                prior = values.iloc[max(0, position - period):position].dropna()
                if prior.empty:
                    raise ValueError(f"Cannot impute {values.index[position].date()}: no prior week is available.")
                values.iloc[position] = prior.mean()
    elif MISSING_METHOD == "linear":
        values = values.interpolate(method="time", limit_area="inside")
    else:
        raise ValueError("Unknown MISSING_METHOD.")
    if values.isna().any():
        raise ValueError("Some calendar-date gaps could not be imputed.")
    history[TARGET_COLUMN] = values.to_numpy()
    return history


def validate_and_prepare(raw):
    raw = raw.copy()
    if MISSING_METHOD not in {"previous_week_mean", "linear"}:
        raise ValueError("MISSING_METHOD must be 'previous_week_mean' or 'linear'.")
    raw.columns = [str(column).strip() for column in raw.columns]
    required = {ITEM_ID_COLUMN, INPUT_DATE_COLUMN, INPUT_TARGET_COLUMN}
    missing = required.difference(raw.columns)
    if missing:
        raise ValueError(f"Missing required input columns: {sorted(missing)}. Found: {list(raw.columns)}")
    raw[DATE_COLUMN] = parse_dates(raw[INPUT_DATE_COLUMN])
    raw[TARGET_COLUMN] = coerce_numeric(raw[INPUT_TARGET_COLUMN])
    raw = raw.dropna(subset=[DATE_COLUMN]).sort_values(DATE_COLUMN).reset_index(drop=True)
    removed_weekend_dates = pd.DatetimeIndex([])
    if DATA_CALENDAR == "business":
        removed_weekend_dates = pd.DatetimeIndex(raw.loc[raw[DATE_COLUMN].dt.dayofweek >= 5, DATE_COLUMN].unique()).sort_values()
        raw = raw[raw[DATE_COLUMN].dt.dayofweek < 5].copy()
    if raw[DATE_COLUMN].duplicated().any():
        examples = raw.loc[raw[DATE_COLUMN].duplicated(), DATE_COLUMN].dt.strftime("%Y-%m-%d").tolist()[:8]
        raise ValueError(f"Duplicate timestamps found: {examples}")
    if raw[ITEM_ID_COLUMN].replace("", np.nan).dropna().nunique() != 1:
        raise ValueError("This univariate notebook requires exactly one non-empty item_id.")
    if raw[TARGET_COLUMN].notna().sum() == 0:
        raise ValueError("The target column contains no numeric observations.")

    sheet_covariates = []
    if DATA_SOURCE == "gsheet" and USE_GSHEET_SHEET_COVARIATES:
        excluded = {ITEM_ID_COLUMN, INPUT_DATE_COLUMN, INPUT_TARGET_COLUMN, DATE_COLUMN, TARGET_COLUMN}
        for column in [column for column in raw.columns if column not in excluded]:
            numeric = coerce_numeric(raw[column])
            if numeric.notna().any():
                raw[column] = numeric
                sheet_covariates.append(column)

    historical_end = raw.loc[raw[TARGET_COLUMN].notna(), DATE_COLUMN].max()
    history = raw[raw[DATE_COLUMN] <= historical_end].copy()
    supplied_future = raw[raw[DATE_COLUMN] > historical_end].copy()
    if DATA_CALENDAR == "business":
        dates = pd.bdate_range(history[DATE_COLUMN].min(), historical_end)
        history = history.set_index(DATE_COLUMN).reindex(dates).rename_axis(DATE_COLUMN).reset_index()
        if history[TARGET_COLUMN].isna().any():
            history = impute_missing_history(history)
    elif DATA_CALENDAR == "natural":
        dates = pd.date_range(history[DATE_COLUMN].min(), historical_end, freq="D")
        history = history.set_index(DATE_COLUMN).reindex(dates).rename_axis(DATE_COLUMN).reset_index()
        if history[TARGET_COLUMN].isna().any():
            history = impute_missing_history(history)
    else:
        raise ValueError("DATA_CALENDAR must be 'natural' or 'business'.")

    data = pd.concat([history, supplied_future], ignore_index=True, sort=False).sort_values(DATE_COLUMN)
    data[PROCESSED_TARGET_COLUMN] = np.nan
    observed = data[TARGET_COLUMN].notna()
    data.loc[observed, PROCESSED_TARGET_COLUMN] = transform_target(data.loc[observed, TARGET_COLUMN])
    return data.reset_index(drop=True), sheet_covariates, historical_end, removed_weekend_dates


def make_model_dates(start, end):
    dates = pd.date_range(pd.Timestamp(start).normalize(), pd.Timestamp(end).normalize(), freq="D")
    return dates if DATA_CALENDAR == "natural" else dates[dates.dayofweek < 5]


def build_feature_frame(dates, source_data):
    frame = pd.DataFrame({DATE_COLUMN: pd.DatetimeIndex(dates)})
    if SOURCE_COVARIATE_COLUMNS:
        known = source_data[[DATE_COLUMN, *SOURCE_COVARIATE_COLUMNS]].drop_duplicates(DATE_COLUMN)
        frame = frame.merge(known, on=DATE_COLUMN, how="left")
    return add_date_covariates(frame)


def selected_covariates(feature_frame):
    if not isinstance(BASIC_COVARIATES_TO_INCLUDE, dict):
        raise TypeError("BASIC_COVARIATES_TO_INCLUDE must be a {covariate: True/False} dictionary.")
    basic = [name for name, enabled in BASIC_COVARIATES_TO_INCLUDE.items() if enabled]
    selected = basic + (HOLIDAY_COVARIATES if ADD_HOLIDAY_COVARIATES else [])
    selected += [f"event_{safe_token(name)}" for name in EXTRA_EVENT_DATES]
    selected += list(SERIES_COVARIATES_TO_INCLUDE) + list(SOURCE_COVARIATE_COLUMNS)
    selected = list(dict.fromkeys(selected))
    missing = [name for name in selected if name not in feature_frame.columns]
    if missing:
        raise ValueError(f"Selected covariates were not generated: {missing}")
    return selected


def assert_covariates(train, future, covariates):
    for label, frame in (("train", train), ("future/test", future)):
        missing = [name for name in covariates if name not in frame]
        nulls = [name for name in covariates if name in frame and frame[name].isna().any()]
        non_numeric = [name for name in covariates if name in frame and not pd.api.types.is_numeric_dtype(frame[name])]
        if missing or nulls or non_numeric:
            raise ValueError(f"Invalid {label} covariates: missing={missing}, null={nulls}, non_numeric={non_numeric}")


def covariate_origin(name):
    if name in SOURCE_COVARIATE_COLUMNS:
        return "Google Sheets"
    if name in HOLIDAY_COVARIATES:
        return f"holiday ({HOLIDAY_COUNTRY})"
    if name.startswith("event_"):
        return "special event"
    if name in SERIES_COVARIATES_TO_INCLUDE:
        return "series-specific"
    return "basic date"


def plot_data_diagnostics(data):
    observed = data.dropna(subset=[TARGET_COLUMN])[[DATE_COLUMN, TARGET_COLUMN, PROCESSED_TARGET_COLUMN]]
    fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
    axes[0].plot(observed[DATE_COLUMN], observed[TARGET_COLUMN], color="black", lw=1)
    axes[0].set_title("Target before transformation (level)")
    axes[1].plot(observed[DATE_COLUMN], observed[PROCESSED_TARGET_COLUMN], color="#1b9e77", lw=1)
    axes[1].set_title(f"Target after transformation: {TRANSFORMATION}")
    for axis in axes:
        axis.grid(alpha=.25)
    plt.tight_layout()
    plt.show()

    period, frequency = (7, "D") if DATA_CALENDAR == "natural" else (5, "B")
    series = observed.set_index(DATE_COLUMN)[TARGET_COLUMN].asfreq(frequency)
    if len(series) < 2 * period or series.isna().any():
        print("Seasonal decomposition skipped: the validated series is too short or irregular.")
        return
    decomposition = seasonal_decompose(series, model="additive", period=period, extrapolate_trend="freq")
    fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True)
    parts = [
        (decomposition.observed, "Observed target"), (decomposition.trend, "Trend"),
        (decomposition.seasonal, f"Seasonality (period={period})"), (decomposition.resid, "Residual / noise"),
    ]
    for axis, (values, title) in zip(axes, parts):
        axis.plot(values.index, values.values, lw=1)
        axis.set_title(f"{title} (level)")
        axis.grid(alpha=.25)
    plt.tight_layout()
    plt.show()


In [ ]:
# Chronos-2, rolling evaluation, and plotting functions
_PIPELINE = None


def get_pipeline():
    global _PIPELINE
    if _PIPELINE is not None:
        return _PIPELINE
    configure_huggingface_certificates()
    last_error = None
    for _ in range(2):
        reset_huggingface_session()
        try:
            _PIPELINE = Chronos2Pipeline.from_pretrained(MODEL_NAME, device_map=CHRONOS2_DEVICE_MAP)
            return _PIPELINE
        except Exception as error:
            last_error = error
            message = str(error).lower()
            if not any(token in message for token in ("certificate", "ssl", "client has been closed")):
                raise
    raise RuntimeError(
        "Hugging Face TLS verification failed. Install the corporate CA in Windows or set "
        "HUGGINGFACE_CA_BUNDLE to its PEM file, restart the kernel, and rerun Section (3). "
        "SSL verification has not been disabled."
    ) from last_error


def chronos_forecast(train, future, covariates):
    if not 1 <= len(future) <= 1024:
        raise ValueError(f"Chronos-2 supports 1-1024 forecast steps; received {len(future)}.")
    context = train.tail(CHRONOS2_MAX_CONTEXT).copy().assign(item_id=TARGET_NAME)
    future = future.copy().assign(item_id=TARGET_NAME)
    model_dates = pd.date_range("2000-01-01", periods=len(context) + len(future), freq="D")
    context[DATE_COLUMN], future[DATE_COLUMN] = model_dates[:len(context)], model_dates[len(context):]
    context = context[["item_id", DATE_COLUMN, PROCESSED_TARGET_COLUMN, *covariates]]
    future = future[["item_id", DATE_COLUMN, *covariates]]
    prediction = get_pipeline().predict_df(
        context, future_df=future, prediction_length=len(future), quantile_levels=QUANTILE_LEVELS,
        id_column="item_id", timestamp_column=DATE_COLUMN, target=PROCESSED_TARGET_COLUMN,
        context_length=min(CHRONOS2_MAX_CONTEXT, len(context)), batch_size=CHRONOS2_BATCH_SIZE,
    )
    median_column = next((column for column in prediction if str(column) in {"0.5", "0.50"}), None)
    if median_column is None:
        raise RuntimeError(f"Chronos-2 did not return p50. Columns: {list(prediction.columns)}")
    values = pd.to_numeric(prediction[median_column], errors="coerce").to_numpy(float)
    if len(values) != len(future) or not np.isfinite(values).all():
        raise RuntimeError("Chronos-2 returned invalid forecast values.")
    return values


def forecast_for_dates(data, cutoff, forecast_dates):
    observed = data.dropna(subset=[PROCESSED_TARGET_COLUMN]).copy()
    cutoff = pd.Timestamp(cutoff).normalize()
    valid_cutoffs = observed.loc[observed[DATE_COLUMN] <= cutoff, DATE_COLUMN]
    if valid_cutoffs.empty:
        raise ValueError(f"No target history exists before {cutoff.date()}.")
    cutoff = valid_cutoffs.max()
    train_start = observed[DATE_COLUMN].min()
    train_dates = observed.loc[(observed[DATE_COLUMN] >= train_start) & (observed[DATE_COLUMN] <= cutoff), DATE_COLUMN]
    train = build_feature_frame(train_dates, data).merge(
        observed[[DATE_COLUMN, PROCESSED_TARGET_COLUMN, TARGET_COLUMN]], on=DATE_COLUMN, how="left"
    )
    future = build_feature_frame(forecast_dates, data)
    covariates = selected_covariates(pd.concat([train, future], ignore_index=True, sort=False))
    assert_covariates(train, future, covariates)
    forecast = future[[DATE_COLUMN]].copy()
    forecast["yhat_proc"] = chronos_forecast(train, future, covariates)
    forecast["yhat"] = inverse_transform(forecast["yhat_proc"])
    forecast = forecast.merge(observed[[DATE_COLUMN, TARGET_COLUMN]], on=DATE_COLUMN, how="left")
    return {"cutoff": cutoff, "forecast": forecast, "covariates": covariates, "train": train}


def nth_model_day(month, number):
    month = pd.Timestamp(month).to_period("M").to_timestamp()
    dates = make_model_dates(month, month + pd.offsets.MonthEnd(0))
    return pd.NaT if number > len(dates) else dates[number - 1]


def evaluation_cutoffs():
    days = NATURAL_CUTOFF_DAYS if DATA_CALENDAR == "natural" else BUSINESS_CUTOFF_DAYS
    evaluation_start = pd.Period(EVALUATION_START, freq="M")
    evaluation_end = pd.Period(EVALUATION_END, freq="M")
    last_cutoff_month = evaluation_end - NEXT_FULL_MONTHS
    if last_cutoff_month < evaluation_start:
        raise ValueError("EVALUATION_END must allow all forecast months through m3.")
    last_cutoff = nth_model_day(last_cutoff_month.to_timestamp(), days[0])
    months = pd.period_range(evaluation_start, last_cutoff_month, freq="M").to_timestamp()
    cutoffs = []
    for month in months:
        for week_number, day_number in enumerate(days, start=1):
            cutoff = nth_model_day(month, day_number)
            if pd.notna(cutoff) and cutoff <= min(last_cutoff, HISTORICAL_END):
                cutoffs.append((cutoff, day_number, week_number))
    return cutoffs


def run_evaluation(data):
    summaries, forecasts = [], []
    cutoffs = evaluation_cutoffs()
    print(f"Evaluating all configured windows: {len(cutoffs)}")
    for position, (cutoff, cutoff_day, week_number) in enumerate(cutoffs, start=1):
        print(f"Processing window {position}/{len(cutoffs)}: cutoff {cutoff.date()} (w{week_number})")
        end_month = cutoff.to_period("M").to_timestamp() + pd.DateOffset(months=NEXT_FULL_MONTHS)
        forecast_dates = make_model_dates(cutoff + pd.Timedelta(days=1), end_month + pd.offsets.MonthEnd(0))
        result = forecast_for_dates(data, cutoff, forecast_dates)
        forecast = result["forecast"].copy()
        forecast["cutoff_date"], forecast["cutoff_day"], forecast["week_no"] = cutoff, cutoff_day, week_number
        forecasts.append(forecast)
        scored = forecast.dropna(subset=[TARGET_COLUMN])
        error = scored["yhat"] - scored[TARGET_COLUMN]
        summaries.append({
            "cutoff_date": cutoff, "cutoff_day": cutoff_day, "week_no": week_number,
            "train_start": result["train"][DATE_COLUMN].min(),
            "forecast_start": forecast[DATE_COLUMN].min(), "forecast_end": forecast[DATE_COLUMN].max(),
            "n_steps": len(forecast), "n_observed_steps": len(scored),
            "MAE": error.abs().mean(), "RMSE": np.sqrt(np.mean(error ** 2)),
            "MAPE_pct": (error.abs() / scored[TARGET_COLUMN].replace(0, np.nan)).mean() * 100,
            "covariates": ", ".join(result["covariates"]) or "none",
        })
    return (
        pd.DataFrame(summaries),
        pd.concat(forecasts, ignore_index=True) if forecasts else pd.DataFrame(),
        build_monthly_evaluation(pd.concat(forecasts, ignore_index=True) if forecasts else pd.DataFrame(), data),
    )



def validate_results_scale(scale, setting_name):
    if scale not in {"original", "transformed"}:
        raise ValueError(f"{setting_name} must be 'original' or 'transformed'.")


def results_scale_label(scale):
    validate_results_scale(scale, "results scale")
    return "level" if scale == "original" else f"transf={TRANSFORMATION}"


def daily_results_columns():
    validate_results_scale(DAILY_RESULTS_SCALE, "DAILY_RESULTS_SCALE")
    return ((TARGET_COLUMN, "yhat") if DAILY_RESULTS_SCALE == "original"
            else (PROCESSED_TARGET_COLUMN, "yhat_proc"))


def monthly_results_columns():
    validate_results_scale(MONTHLY_RESULTS_SCALE, "MONTHLY_RESULTS_SCALE")
    return (("real_level", "pred_level") if MONTHLY_RESULTS_SCALE == "original"
            else ("real_proc", "pred_proc"))


def relative_error_pp(predicted, observed):
    return (100 * (predicted - observed) / observed
            if np.isfinite(observed) and observed != 0 and np.isfinite(predicted)
            else np.nan)


def aggregate_monthly_value(values):
    """Aggregate one complete month in original scale using MONTHLY_VALUE."""
    if MONTHLY_VALUE not in {"sum", "mean", "last"}:
        raise ValueError("MONTHLY_VALUE must be 'sum', 'mean', or 'last'.")
    values = pd.to_numeric(pd.Series(values), errors="coerce").dropna()
    if values.empty:
        return np.nan
    if MONTHLY_VALUE == "sum":
        return values.sum()
    if MONTHLY_VALUE == "mean":
        return values.mean()
    return values.iloc[-1]


def build_monthly_evaluation(evaluation_windows, data):
    """Score monthly aggregates exactly as the XLSX evaluation output requires."""
    if evaluation_windows.empty:
        return pd.DataFrame()
    observed = data.dropna(subset=[TARGET_COLUMN])[[DATE_COLUMN, TARGET_COLUMN]].copy()
    rows = []
    group_columns = ["cutoff_date", "cutoff_day", "week_no"]
    for keys, window in evaluation_windows.groupby(group_columns, sort=True):
        cutoff, cutoff_day, week_number = keys
        cutoff = pd.Timestamp(cutoff)
        for period, forecast_part in window.groupby(window[DATE_COLUMN].dt.to_period("M"), sort=True):
            period_start = period.to_timestamp()
            period_end = period.to_timestamp("M")
            period_dates = make_model_dates(period_start, period_end)
            actual_period = observed[observed[DATE_COLUMN].isin(period_dates)]
            partial_actual = actual_period[actual_period[DATE_COLUMN] <= cutoff]
            available_dates = set(actual_period.loc[actual_period[TARGET_COLUMN].notna(), DATE_COLUMN])
            full_observed = len(period_dates) > 0 and set(period_dates).issubset(available_dates)
            real_level = aggregate_monthly_value(actual_period[TARGET_COLUMN]) if full_observed else np.nan
            pred_level = aggregate_monthly_value(pd.concat([
                partial_actual[TARGET_COLUMN], forecast_part["yhat"]
            ], ignore_index=True))
            real_proc = transform_target([real_level])[0] if np.isfinite(real_level) else np.nan
            pred_proc = transform_target([pred_level])[0] if np.isfinite(pred_level) else np.nan
            error_pp_level = relative_error_pp(pred_level, real_level)
            error_pp_proc = relative_error_pp(pred_proc, real_proc)
            error_pp = error_pp_level if MONTHLY_RESULTS_SCALE == "original" else error_pp_proc
            offset = (period_start.year - cutoff.year) * 12 + period_start.month - cutoff.month
            rows.append({
                "cutoff_date": cutoff, "cutoff_day": cutoff_day, "week_no": week_number,
                "period_start": period_start, "period_end": period_end,
                "period_offset": int(offset), "real_level": real_level, "pred_level": pred_level,
                "real_proc": real_proc, "pred_proc": pred_proc,
                "error_pp_level": error_pp_level, "error_pp_proc": error_pp_proc,
                "error_pp": error_pp, "abs_error_pp": abs(error_pp), "forecast_n": len(forecast_part),
                "expected_n": len(period_dates), "has_full_observed_period": full_observed,
            })
    return pd.DataFrame(rows).sort_values(["cutoff_date", "period_start"]).reset_index(drop=True)


def score_monthly(monthly_evaluation):
    valid = monthly_evaluation[
        monthly_evaluation["has_full_observed_period"] &
        monthly_evaluation["error_pp"].notna() &
        monthly_evaluation["period_offset"].between(0, NEXT_FULL_MONTHS)
    ].copy()
    if valid.empty:
        return pd.DataFrame()
    valid["_squared_error"] = valid["error_pp"] ** 2
    stats = valid.groupby(["week_no", "period_offset"], as_index=False).agg(
        n_periods=("error_pp", "size"), ME_pp=("error_pp", "mean"),
        MAE_pp=("abs_error_pp", "mean"), MedAE_pp=("abs_error_pp", "median"),
        _mse=("_squared_error", "mean"),
    )
    stats["RMSE_pp"] = np.sqrt(stats.pop("_mse"))
    stats["results_scale"] = results_scale_label(MONTHLY_RESULTS_SCALE)
    return stats.rename(columns={"week_no": "cutoff_week", "period_offset": "pred_month"})


def build_output_xlsx_path(base_name="chronos2", output_dir="output_chronos"):
    def token(value):
        return re.sub(r"[^0-9A-Za-z_.-]+", "-", str(value).strip().lower()).strip("-") or "none"
    return str(Path(output_dir) / (
        f"{token(MODEL_NAME)}_{token(TARGET_NAME)}_transf-{token(TRANSFORMATION)}_monthly-{token(MONTHLY_VALUE)}_calendar-{token(DATA_CALENDAR)}_{EVALUATION_START[:4]}-{EVALUATION_END[:4]}.xlsx"
    ))

def export_evaluation_xlsx(summary, windows, monthly_evaluation, output_xlsx):
    window_rows, prediction_frames = [], []
    for window_number, (cutoff, window) in enumerate(windows.groupby("cutoff_date", sort=True), start=1):
        window = window.sort_values(DATE_COLUMN)
        true_column, predicted_column = daily_results_columns()
        prediction = window[[DATE_COLUMN, true_column, predicted_column]].copy()
        scale_label = results_scale_label(DAILY_RESULTS_SCALE)
        prediction.columns = ["fecha", f"w{window_number}_true_{scale_label}", f"w{window_number}_pred_{scale_label}"]
        prediction_frames.append(prediction)
        window_months = monthly_evaluation[monthly_evaluation["cutoff_date"] == cutoff]
        summary_row = summary[summary["cutoff_date"] == cutoff].iloc[0]
        for _, month in window_months.iterrows():
            window_rows.append({
                "evaluation_start_month": EVALUATION_START, "evaluation_end_month": EVALUATION_END,
                "next_full_months": NEXT_FULL_MONTHS,
                "window": window_number, "cutoff_week": int(month["week_no"]),
                "cutoff_date": pd.Timestamp(cutoff).strftime("%Y-%m-%d"),
                "pred_month": int(month["period_offset"]),
                "train_start": pd.Timestamp(summary_row["train_start"]).strftime("%Y-%m-%d"),
                "test_start": pd.Timestamp(month["period_start"]).strftime("%Y-%m-%d"),
                "test_end": pd.Timestamp(month["period_end"]).strftime("%Y-%m-%d"),
                "horizon_start": pd.Timestamp(summary_row["forecast_start"]).strftime("%Y-%m-%d"),
                "horizon_end": pd.Timestamp(summary_row["forecast_end"]).strftime("%Y-%m-%d"),
                "prediction_length": int(summary_row["n_steps"]),
                "real_level": month["real_level"], "pred_level": month["pred_level"],
                "real_proc": month["real_proc"], "pred_proc": month["pred_proc"],
                "error_pp_level": month["error_pp_level"], "error_pp_proc": month["error_pp_proc"],
                "abs_error_pp": month["abs_error_pp"], "full_month_obs": month["has_full_observed_period"],
            })
    selected_covariate_names = summary.iloc[0]["covariates"] if not summary.empty else "none"
    evaluation_start = windows["cutoff_date"].min() if not windows.empty else pd.NaT
    evaluation_end = windows["cutoff_date"].max() if not windows.empty else pd.NaT
    metadata = pd.DataFrame([{
        "model_id": MODEL_NAME, "transformation": TRANSFORMATION, "target": TARGET_NAME,
        "calendar": DATA_CALENDAR, "missing_method": MISSING_METHOD,
        "batch_size": CHRONOS2_BATCH_SIZE, "context_length": CHRONOS2_MAX_CONTEXT,
        "quantile": ", ".join(map(str, QUANTILE_LEVELS)), "covariates": selected_covariate_names,
        "monthly_value": MONTHLY_VALUE,
        "daily_results_scale": results_scale_label(DAILY_RESULTS_SCALE),
        "monthly_results_scale": results_scale_label(MONTHLY_RESULTS_SCALE),
        "monthly_metric_scale": f"{results_scale_label(MONTHLY_RESULTS_SCALE)}; relative percentage points",
        "holidays": HOLIDAY_COUNTRY if ADD_HOLIDAY_COVARIATES else "none",
        "evaluation_start": pd.Timestamp(evaluation_start).strftime("%Y-%m-%d") if pd.notna(evaluation_start) else None,
        "evaluation_end": pd.Timestamp(evaluation_end).strftime("%Y-%m-%d") if pd.notna(evaluation_end) else None,
        "evaluation_windows": int(windows["cutoff_date"].nunique()) if not windows.empty else 0,
    }])
    windows_output = pd.DataFrame(window_rows, columns=[
        "evaluation_start_month", "evaluation_end_month", "next_full_months",
        "window", "cutoff_week", "cutoff_date", "pred_month", "train_start",
        "test_start", "test_end", "horizon_start", "horizon_end", "prediction_length",
        "real_level", "pred_level", "real_proc", "pred_proc", "error_pp_level", "error_pp_proc", "abs_error_pp", "full_month_obs",
    ])
    if prediction_frames:
        predictions = prediction_frames[0]
        for frame in prediction_frames[1:]:
            predictions = predictions.merge(frame, on="fecha", how="outer")
        predictions = predictions.sort_values("fecha")
        predictions["fecha"] = pd.to_datetime(predictions["fecha"]).dt.strftime("%Y-%m-%d")
    else:
        predictions = pd.DataFrame()
    statistics = score_monthly(monthly_evaluation)
    output = Path(output_xlsx)
    output.parent.mkdir(parents=True, exist_ok=True)
    with pd.ExcelWriter(output, engine="openpyxl") as writer:
        metadata.to_excel(writer, sheet_name="metadata", index=False)
        windows_output.to_excel(writer, sheet_name="windows", index=False)
        predictions.to_excel(writer, sheet_name="predictions", index=False)
        statistics.to_excel(writer, sheet_name="statistics", index=False)
    return str(output), metadata, windows_output, predictions, statistics


def plot_monthly_mae_pp(monthly_evaluation):
    statistics = score_monthly(monthly_evaluation)
    if statistics.empty:
        print("No fully observed monthly periods are available for the MAE_pp plot.")
        return
    statistics = statistics.sort_values(["cutoff_week", "pred_month"])
    labels = [f"w{week}_m{month}" for week, month in zip(statistics["cutoff_week"], statistics["pred_month"])]
    plt.figure(figsize=(max(12, len(labels) * 0.55), 5))
    plt.bar(labels, statistics["MAE_pp"], color="#1b9e77")
    plt.ylabel(f"Mean MAE_pp ({results_scale_label(MONTHLY_RESULTS_SCALE)})")
    plt.xlabel("Evaluation window and forecast month")
    plt.title(f"Mean MAE_pp by window and month ({results_scale_label(MONTHLY_RESULTS_SCALE)})")
    plt.xticks(rotation=45, ha="right")
    plt.grid(axis="y", alpha=.25)
    plt.tight_layout()
    plt.show()

def plot_monthly_medae_pp(monthly_evaluation):
    statistics = score_monthly(monthly_evaluation)
    if statistics.empty:
        print("No fully observed monthly periods are available for the MedAE_pp plot.")
        return
    statistics = statistics.sort_values(["cutoff_week", "pred_month"])
    labels = [f"w{week}_m{month}" for week, month in zip(statistics["cutoff_week"], statistics["pred_month"])]
    plt.figure(figsize=(max(12, len(labels) * 0.55), 5))
    plt.bar(labels, statistics["MedAE_pp"], color="#1b9e77")
    plt.ylabel(f"Mean MedAE_pp ({results_scale_label(MONTHLY_RESULTS_SCALE)})")
    plt.xlabel("Evaluation window and forecast month")
    plt.title(f"Mean MedAE_pp by window and month ({results_scale_label(MONTHLY_RESULTS_SCALE)})")
    plt.xticks(rotation=45, ha="right")
    plt.grid(axis="y", alpha=.25)
    plt.tight_layout()
    plt.show()


def plot_monthly_actual_vs_predicted(monthly_evaluation):
    """Plot every window-level monthly forecast against the unique observed monthly series."""
    real_column, predicted_column = monthly_results_columns()
    valid = monthly_evaluation[
        monthly_evaluation["has_full_observed_period"] &
        monthly_evaluation[real_column].notna() &
        monthly_evaluation[predicted_column].notna() &
        monthly_evaluation["period_offset"].between(0, NEXT_FULL_MONTHS)
    ].copy()
    if valid.empty:
        print("No fully observed monthly periods are available for the monthly series plot.")
        return
    valid = valid.sort_values(["cutoff_date", "period_start"])
    actual = valid.groupby("period_start", as_index=False)[real_column].first()
    plt.figure(figsize=(14, 6))
    first = True
    for _, trajectory in valid.groupby("cutoff_date", sort=True):
        plt.plot(
            trajectory["period_start"], trajectory[predicted_column],
            color="#377eb8", alpha=.16, linewidth=1,
            label="Predicted (each cutoff window)" if first else None,
        )
        first = False
    plt.plot(actual["period_start"], actual[real_column], color="black", linewidth=2,
             marker="o", markersize=3, label="Actual")
    scale_label = results_scale_label(MONTHLY_RESULTS_SCALE)
    plt.ylabel(f"Monthly value ({MONTHLY_VALUE}; {scale_label})")
    plt.xlabel("Month")
    plt.title(f"Monthly actual vs predicted values ({scale_label})")
    plt.grid(alpha=.25)
    plt.legend()
    plt.tight_layout()
    plt.show()


def plot_monthly_rmse_pp_grid(monthly_evaluation):
    """Show RMSE_pp by cutoff week in separate panels for m0, m1, m2, and m3."""
    statistics = score_monthly(monthly_evaluation)
    if statistics.empty:
        print("No fully observed monthly periods are available for the RMSE_pp grid.")
        return
    fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharey=True)
    weeks = np.arange(1, 5)
    for pred_month, ax in enumerate(axes.flat):
        panel = statistics[statistics["pred_month"] == pred_month].set_index("cutoff_week")
        values = panel["RMSE_pp"].reindex(weeks)
        bars = ax.bar([f"w{week}" for week in weeks], values, color="#7570b3")
        ax.set_title(f"m{pred_month}")
        ax.set_xlabel("Cutoff week")
        ax.set_ylabel(f"RMSE (pp; {results_scale_label(MONTHLY_RESULTS_SCALE)})")
        ax.grid(axis="y", alpha=.25)
        for bar, value in zip(bars, values):
            if pd.notna(value):
                ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(), f"{value:.2f}",
                        ha="center", va="bottom", fontsize=9)
    fig.suptitle(f"Monthly RMSE by cutoff week and forecast month ({MONTHLY_VALUE}; {results_scale_label(MONTHLY_RESULTS_SCALE)})")
    fig.tight_layout()
    plt.show()


def selected_plot_cutoffs(evaluation_windows):
    available = pd.DatetimeIndex(evaluation_windows["cutoff_date"].drop_duplicates()).sort_values()
    selected = []
    for year, month in PLOT_EVALUATION_YEAR_MONTHS:
        candidates = available[(available.year == year) & (available.month == month)]
        if len(candidates):
            selected.append(candidates[0])
    if len(selected) < N_EVALUATION_PLOTS and len(available):
        selected.extend(available[index] for index in np.linspace(0, len(available) - 1, N_EVALUATION_PLOTS, dtype=int))
    return list(dict.fromkeys(selected))[:N_EVALUATION_PLOTS]


def plot_evaluation_windows(data, evaluation_windows):
    selected = selected_plot_cutoffs(evaluation_windows)
    if not selected:
        print("No evaluation windows are available to plot.")
        return
    rows = int(np.ceil(len(selected) / 2))
    fig, axes = plt.subplots(rows, 2, figsize=(16, 3.8 * rows), squeeze=False)
    for axis, cutoff in zip(axes.ravel(), selected):
        forecast = evaluation_windows[evaluation_windows["cutoff_date"] == cutoff]
        actual = data[(data[DATE_COLUMN] >= cutoff - pd.Timedelta(days=45)) &
                      (data[DATE_COLUMN] <= forecast[DATE_COLUMN].max())]
        true_column, predicted_column = daily_results_columns()
        scale_label = results_scale_label(DAILY_RESULTS_SCALE)
        axis.plot(actual[DATE_COLUMN], actual[true_column], color="black", lw=1.2, label="actual")
        axis.plot(forecast[DATE_COLUMN], forecast[predicted_column], color="#d95f02", lw=2, label="p50 forecast")
        axis.axvline(cutoff, color="grey", ls="--", lw=1, label="training cutoff")
        axis.set_title(f"Evaluation window ({scale_label}): {pd.Timestamp(cutoff).date()}")
        axis.set_ylabel(f"Value ({scale_label})")
        axis.legend(fontsize=8)
        axis.grid(alpha=.25)
    for axis in axes.ravel()[len(selected):]:
        axis.set_visible(False)
    plt.tight_layout()


## (4) Data import and special dummies

Configure and define the special dummies here, then import the data and apply all selected dummies.


### (4.a) Special dummy configuration — **user editable**

Add one-off event dates to `EXTRA_EVENT_DATES`. For recurrent or formula-based dummies, edit `add_series_specific_covariates` and list the resulting columns in `SERIES_COVARIATES_TO_INCLUDE`.


In [ ]:
HOLIDAY_COVARIATES = ["is_holiday", "is_pre_holiday", "is_post_holiday", "is_bridge_day"]


In [ ]:
# ===== USER-EDITABLE SERIES-SPECIFIC COVARIATES =====
EXTRA_EVENT_DATES = {
    "black_friday": ["2018-11-23", "2019-11-29", "2020-11-27", "2021-11-26", "2022-11-25", "2023-11-24", "2024-11-29", "2025-11-28"],
}

SERIES_COVARIATES_TO_INCLUDE = [
    # "is_campaign",
]


def add_series_specific_covariates(df):
    df = df.copy()
    ts = pd.to_datetime(df["timestamp"])

    # Example:
    # df["is_campaign"] = ((ts.dt.month == 8) & (ts.dt.day <= 15)).astype(int)

    return df
# ====================================================


In [ ]:
def get_holiday_calendar(country):
    value = str(country).strip()
    normalized = value.casefold()
    if normalized == "spain":
        from workalendar.europe import Spain
        return Spain()
    if normalized in {"united states", "united states of america", "usa"}:
        from workalendar.usa import UnitedStates
        return UnitedStates()
    from workalendar.registry import registry
    aliases = {
        "france": "FR", "germany": "DE", "italy": "IT", "portugal": "PT",
        "united kingdom": "GB", "uk": "GB", "canada": "CA", "australia": "AU",
    }
    calendar_class = registry.get(aliases.get(normalized, value.upper()))
    if calendar_class is None:
        raise ValueError(f"Unsupported HOLIDAY_COUNTRY: {country!r}. Use a supported name or ISO code.")
    return calendar_class()


def holiday_dates(years):
    if not ADD_HOLIDAY_COVARIATES:
        return pd.DatetimeIndex([])
    calendar = get_holiday_calendar(HOLIDAY_COUNTRY)
    return pd.DatetimeIndex([
        pd.Timestamp(day) for year in years for day, _ in calendar.holidays(int(year))
    ]).normalize()


def add_date_covariates(df):
    df = df.copy()
    ts = pd.to_datetime(df[DATE_COLUMN])
    normalized = ts.dt.normalize()
    day_of_week = ts.dt.dayofweek
    month_start = normalized.dt.to_period("M").dt.to_timestamp()
    month_end = normalized.dt.to_period("M").dt.to_timestamp("M")
    first_business_day = month_start + pd.to_timedelta((7 - month_start.dt.dayofweek) % 7, unit="D")
    last_business_day = month_end - pd.to_timedelta((month_end.dt.dayofweek - 4) % 7, unit="D")

    df["day_of_week"] = day_of_week.astype(int)
    df["day_of_month"] = ts.dt.day.astype(int)
    df["first_natural_day_of_month"] = normalized.eq(month_start).astype(int)
    df["first_business_day_of_month"] = normalized.eq(first_business_day).astype(int)
    df["last_natural_day_of_month"] = normalized.eq(month_end).astype(int)
    df["last_business_day_of_month"] = normalized.eq(last_business_day).astype(int)
    angle = 2 * np.pi * ts.dt.month / 12
    df["month_sin"] = np.sin(angle)
    df["month_cos"] = np.cos(angle)
    df["weekend"] = (day_of_week >= 5).astype(int)
    if ADD_HOLIDAY_COVARIATES:
        holidays = holiday_dates(sorted(ts.dt.year.dropna().unique()))
        df["is_holiday"] = normalized.isin(holidays).astype(int)
        df["is_pre_holiday"] = (normalized + pd.Timedelta(days=1)).isin(holidays).astype(int)
        df["is_post_holiday"] = (normalized - pd.Timedelta(days=1)).isin(holidays).astype(int)
        previous_holiday = (normalized - pd.Timedelta(days=1)).isin(holidays)
        next_holiday = (normalized + pd.Timedelta(days=1)).isin(holidays)
        df["is_bridge_day"] = (
            (day_of_week < 5) & ~normalized.isin(holidays) &
            ((previous_holiday & (day_of_week == 4)) | (next_holiday & (day_of_week == 0)))
        ).astype(int)
    for name, dates in EXTRA_EVENT_DATES.items():
        event_dates = pd.DatetimeIndex(pd.to_datetime(list(dates))).normalize()
        df[f"event_{safe_token(name)}"] = normalized.isin(event_dates).astype(int)
    return add_series_specific_covariates(df)

### (4.b) Data import, validation and dummy application


In [ ]:
df_raw = load_raw_input()
Data, SOURCE_COVARIATE_COLUMNS, HISTORICAL_END, REMOVED_WEEKEND_DATES = validate_and_prepare(df_raw)
if DATA_CALENDAR == "business" and len(REMOVED_WEEKEND_DATES):
    examples = REMOVED_WEEKEND_DATES.strftime("%Y-%m-%d").tolist()[:8]
    print(f"DISCLAIMER: Removed {len(REMOVED_WEEKEND_DATES)} Saturday/Sunday observation(s) because the business calendar contains Monday-Friday only. Examples: {examples}")

diagnostic_features = build_feature_frame(Data[DATE_COLUMN].drop_duplicates(), Data)
SELECTED_COVARIATES = selected_covariates(diagnostic_features)
covariate_summary = pd.DataFrame({
    "covariate": SELECTED_COVARIATES,
    "origin": [covariate_origin(name) for name in SELECTED_COVARIATES],
})

print(f"Rows: {len(Data):,}")
print(f"Observed period: {Data.loc[Data[TARGET_COLUMN].notna(), DATE_COLUMN].min().date()} to {HISTORICAL_END.date()}")
print("Selected covariates:")
display(covariate_summary if len(covariate_summary) else pd.DataFrame({"covariate": ["none"], "origin": ["-"]}))
plot_data_diagnostics(Data)


## (5) Evaluation


In [ ]:
# EVALUATION CONFIGURATION
# The training set always starts at the first date available in the dataframe.
# EVALUATION_END is the final evaluated month (m3 for the final cutoff).
EVALUATION_START = "2022-01"
EVALUATION_END = "2025-12"
NEXT_FULL_MONTHS = 3
MONTHLY_VALUE = "sum"                       # "sum", "mean", or "last"
DAILY_RESULTS_SCALE = "original"             # "original" (level) or "transformed" (transf=<TRANSFORMATION>)
MONTHLY_RESULTS_SCALE = "original"           # "original" (level) or "transformed" (transf=<TRANSFORMATION>)
N_EVALUATION_PLOTS = 24                       # Plot subset only
PLOT_EVALUATION_YEAR_MONTHS = [
    (2022, 1), (2022, 2), (2022, 3), (2022, 4), (2022, 5), (2022, 6), (2022, 7), (2022, 8), (2022, 9), (2022, 10), (2022, 11), (2022, 12),
    (2025, 1), (2025, 2), (2025, 3), (2025, 4), (2025, 5), (2025, 6), (2025, 7), (2025, 8), (2025, 9), (2025, 10), (2025, 11), (2025, 12)
]

In [ ]:
evaluation_summary, evaluation_windows, evaluation_monthly = run_evaluation(Data)
display(evaluation_summary)
monthly_comparison = score_monthly(evaluation_monthly)
if not monthly_comparison.empty:
    display(monthly_comparison)

OUTPUT_XLSX = build_output_xlsx_path()
_, metadata_output, windows_output, predictions_output, statistics_output = export_evaluation_xlsx(
    evaluation_summary, evaluation_windows, evaluation_monthly, OUTPUT_XLSX
)
print("Saved:", OUTPUT_XLSX)
plot_monthly_mae_pp(evaluation_monthly)
plot_monthly_medae_pp(evaluation_monthly)
plot_monthly_actual_vs_predicted(evaluation_monthly)
plot_monthly_rmse_pp_grid(evaluation_monthly)
plot_evaluation_windows(Data, evaluation_windows)


## (6) Production forecast


In [ ]:
# FORECAST CONFIGURATION
CUTOFF_DATE = "2025-12-01"                # None uses all observed history
FORECAST_DAYS = 28

production_cutoff = HISTORICAL_END if CUTOFF_DATE is None else pd.Timestamp(CUTOFF_DATE)
production_dates = make_model_dates(production_cutoff + pd.Timedelta(days=1),
                                    production_cutoff + pd.Timedelta(days=FORECAST_DAYS))
production_result = forecast_for_dates(Data, production_cutoff, production_dates)
production_forecast = production_result["forecast"]
print("Effective cutoff:", production_result["cutoff"].date())
print("Selected covariates:", production_result["covariates"] or "none")
display(production_forecast)


In [ ]:
cutoff = production_result["cutoff"]
forecast_end = production_forecast[DATE_COLUMN].max()
actual = Data[(Data[DATE_COLUMN] >= cutoff - pd.Timedelta(days=90)) &
              (Data[DATE_COLUMN] <= max(HISTORICAL_END, forecast_end))]

plt.figure(figsize=(14, 5))
plt.plot(actual[DATE_COLUMN], actual[TARGET_COLUMN], color="black", lw=1.5, label="actual series")
plt.plot(production_forecast[DATE_COLUMN], production_forecast["yhat"], color="#d95f02", lw=2, label="p50 forecast")
plt.axvline(cutoff, color="grey", ls="--", lw=1, label="training cutoff")
plt.title("Chronos-2 forecast (level)" if cutoff < HISTORICAL_END else "Production forecast - future (level)")
plt.legend()
plt.grid(alpha=.25)
plt.tight_layout()
plt.show()
